### Cell 1 — Runtime Check

In [ ]:
# Verify GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

### Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adapter weights will be saved here after training
DRIVE_SAVE_PATH = "/content/drive/MyDrive/lora-image-classifier/checkpoints"
import os
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
print(f"Drive mounted. Weights will be saved to: {DRIVE_SAVE_PATH}")

### Cell 3 — Clone Repository

In [ ]:
!git clone https://github.com/anantha037/lora-image-classifier.git
%cd lora-image-classifier

### Cell 4 — Install Dependencies

In [ ]:
!pip install -r requirements.txt

import os
os.kill(os.getpid(), 9)

### Cell 5 — Reconnect after restart (re-run from here after restart)

In [ ]:
import os
%cd /content/lora-image-classifier
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE_PATH = "/content/drive/MyDrive/lora-image-classifier/checkpoints"
print("Ready to continue.")

### Cell 6 — Kaggle API Setup

In [ ]:
# Upload your kaggle.json manually using this cell
from google.colab import files
files.upload()  # Upload kaggle.json here

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("Kaggle API configured.")

### Cell 7 — Download Dataset

In [ ]:
!python src/download_dataset.py

# Verify dataset structure
import os
from config import Config
config = Config()
found_classes = 0
for cls in config.SELECTED_CLASSES:
    path = os.path.join(config.DATASET_ROOT, cls)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"{cls}: {count} images")
    if count > 0:
        found_classes += 1

if found_classes < 10:
    raise RuntimeError(f"Verification failed: Expected 10 classes, but found {found_classes}. Check dataset download.")
print("\nDataset verification passed!")


### Cell 8 — Train

In [ ]:
from src.train import train
from config import Config
config = Config()
history = train(config)

### Cell 9 — Display Training Curves

In [ ]:
from IPython.display import Image
Image("outputs/training_curves.png")

### Cell 10 — Run Evaluation

In [ ]:
from src.evaluate import run_evaluation
from config import Config
config = Config()
run_evaluation(config)

### Cell 11 — Display Evaluation Outputs

In [ ]:
from IPython.display import Image, display
print("Confusion Matrix:")
display(Image("outputs/confusion_matrix.png"))
print("Misclassified Samples:")
display(Image("outputs/misclassified_samples.png"))

### Cell 12 — Save Adapter Weights to Google Drive

In [ ]:
import shutil
shutil.copytree("outputs/checkpoints", DRIVE_SAVE_PATH, dirs_exist_ok=True)
print(f"Adapter weights saved to Google Drive at: {DRIVE_SAVE_PATH}")

# Also save training curves and eval outputs
for fname in ["training_curves.png", "confusion_matrix.png", "misclassified_samples.png"]:
    src = f"outputs/{fname}"
    dst = f"/content/drive/MyDrive/lora-image-classifier/{fname}"
    shutil.copy(src, dst)
    print(f"Saved: {fname}")

### Cell 13 — Download Weights Locally (Alternative to Drive)

In [ ]:
# If you prefer direct download instead of Drive
from google.colab import files
import zipfile

with zipfile.ZipFile("adapter_weights.zip", "w") as zf:
    for f in os.listdir("outputs/checkpoints"):
        zf.write(f"outputs/checkpoints/{f}")

files.download("adapter_weights.zip")